In [4]:
import torch

# shape: [2, 3] + [3] -> [2, 3] + [1, 3] -> [2, 3]
A = torch.randn(2, 3)
b = torch.randn(3)
result = A + b  # b 自动广播到 [2, 3]

# Attention 中的 mask 广播
# scores: [batch, heads, seq_len, seq_len]
# mask:   [1, 1, seq_len, seq_len]  -> 广播到 4 维

In [5]:
import torch
import torch.nn.functional as F

# 从零实现交叉熵
def cross_entropy(p, q):
    """p 和 q 都是相同维度的概率分布"""
    return (-p * torch.log(q)).sum()

# 预测不准确时，交叉熵较大
q = torch.tensor([0.3, 0.5, 0.2])  # 预测分布
p = torch.tensor([0.0, 1.0, 0.0])  # 真实分布（类别 1）
print(cross_entropy(p, q))  # tensor(0.6931)

# 预测较准确时，交叉熵较小
q = torch.tensor([0.05, 0.9, 0.05])
print(cross_entropy(p, q))  # tensor(0.1054)

tensor(0.6931)
tensor(0.1054)


In [6]:
# 批量版本，仿 PyTorch 实现
def cross_entropy_with_batch_logits(label, logits):
    """
    label size : [bs]
    logits size: [bs, classes]
    """
    bs, _ = logits.shape
    prob = F.softmax(logits, dim=-1)
    idx = torch.arange(0, bs)
    logprob = prob[idx, label].log()
    CE_loss = -logprob.mean()
    return CE_loss

# 验证与 PyTorch 一致
loss_fn = torch.nn.CrossEntropyLoss()
logits = torch.randn(4, 10)
label = torch.randint(high=10, size=(4,))
print(cross_entropy_with_batch_logits(label, logits))
print(loss_fn(logits, label))  # 结果一致

tensor(2.5765)
tensor(2.5765)


In [7]:
import torch

def SoftMax(logits):
    logits_max,_=logits.max(dim=-1)
    logits=logits-logits_max.unsqueeze(1)
    logits=logits.exp()
    logits_sum=logits.sum(-1,keepdim=True)
    pro=logits/logits_sum
    return pro

logits=torch.randn(8,10)
prob=SoftMax(logits)
print(prob[0].sum())  # 每一行的概率和为 1


tensor(1.0000)


In [8]:
# softmax 后再 log，数值不稳定
logits = torch.tensor([[10, 2, 10000, 4]], dtype=torch.float32)
prob = SoftMax(logits)
print(prob.log())  # tensor([[-inf, -inf, 0., -inf]])  ← 出现 -inf！

# LogSoftmax 直接计算，数值稳定
# log_softmax(x_i) = x_i - c - log(sum(exp(x_j - c)))
print(torch.nn.functional.log_softmax(logits, dim=-1))
# tensor([[-9990., -9998., 0., -9996.]])  ← 正确结果

tensor([[-inf, -inf, 0., -inf]])
tensor([[-9990., -9998.,     0., -9996.]])


In [9]:
def LogSoftMax(logits):
    """数值稳定的 LogSoftmax"""
    logits_max, _ = logits.max(dim=-1)
    safe_logits = logits - logits_max.unsqueeze(1)
    safe_logits_exp = safe_logits.exp()
    safe_logits_sum = safe_logits_exp.sum(-1, keepdim=True)
    log_logits_sum = safe_logits_sum.log()
    log_probs = safe_logits - log_logits_sum
    return log_probs